# Streaming monitor

`StreamMonitor` is the production primitive: feed it batches as they arrive and it scores each
against a moving baseline. This notebook shows the three knobs you'll actually tune:

- **`baseline_strategy`** — fixed, sliding window, or EWMA. Sliding keeps the reference fresh.
- **`on_schema_change`** — `strict` raises on schema drift; `drop` quietly intersects columns.
- **`on_drift`** — a callback fires when drift is flagged so you can route alerts.

Jupyter kernels run inside an event loop, so we use top-level `await` directly
(no `asyncio.run`) when consuming the async iterator.

In [ ]:
import numpy as np
import pandas as pd

from drift_control import StreamMonitor
from drift_control.psi_drift_detector import PSIDriftDetector

rng = np.random.default_rng(0)
baseline = pd.DataFrame(
    {
        "x": rng.normal(0, 1, size=256),
        "y": rng.normal(0, 1, size=256),
    }
)
print("baseline shape:", baseline.shape)

## 1. Fixed baseline + drift callback

The simplest production setup: the baseline is your reference dataset and stays put.
`on_drift` is invoked with the per-column results whenever any column is flagged.

In [ ]:
drift_log: list[dict] = []


def on_drift(result):
    drift_log.append({k: v for k, v in result.items() if v.get("drift")})


monitor = StreamMonitor(
    detector=PSIDriftDetector(threshold=0.2),
    on_drift=on_drift,
)
monitor.set_baseline(baseline)


async def stream(n_batches=20):
    for i in range(n_batches):
        # Inject a mean shift on x after batch 7.
        shift = 0.0 if i < 7 else 0.9
        yield pd.DataFrame(
            {
                "x": rng.normal(shift, 1, size=64),
                "y": rng.normal(0, 1, size=64),
            }
        )


flags = []
async for result in monitor.monitor(stream()):
    flags.append(any(v.get("drift") for v in result.values()))

print(f"batches: {len(flags)}, drift events: {sum(flags)}, callback invocations: {len(drift_log)}")

## 2. Sliding-window re-baselining

When the upstream distribution drifts gradually, a static baseline becomes a false-positive
machine. `baseline_strategy="sliding"` re-fits the baseline from the last N batches so the
detector adapts.

Here we use a 4-batch window and feed a slow drift. After enough batches the baseline catches up
and drift events stop firing even though the upstream is still shifted.

In [ ]:
monitor = StreamMonitor(
    detector=PSIDriftDetector(threshold=0.2),
    baseline_strategy="sliding",
    sliding_window_batches=4,
)
monitor.set_baseline(baseline)


async def slow_drift_stream(n_batches=30):
    for i in range(n_batches):
        yield pd.DataFrame(
            {
                "x": rng.normal(loc=0.05 * i, scale=1, size=64),
                "y": rng.normal(0, 1, size=64),
            }
        )


flags = []
async for result in monitor.monitor(slow_drift_stream()):
    flags.append(any(v.get("drift") for v in result.values()))

early = sum(flags[:10])
late = sum(flags[-10:])
print(f"drift events early (first 10): {early}, late (last 10): {late}")
print("sliding window quiets the late phase because the baseline catches up.")

## 3. Schema evolution

Production schemas change. `on_schema_change="drop"` intersects baseline and batch columns
so a new column doesn't crash the monitor (it's just ignored for scoring). `strict` (the default)
raises so you find out about schema drift immediately.

In [ ]:
monitor = StreamMonitor(
    detector=PSIDriftDetector(threshold=0.2),
    on_schema_change="drop",
)
monitor.set_baseline(baseline)


async def schema_change_stream():
    yield pd.DataFrame({"x": rng.normal(0, 1, size=64), "y": rng.normal(0, 1, size=64)})
    # New column appears upstream.
    yield pd.DataFrame(
        {
            "x": rng.normal(0, 1, size=64),
            "y": rng.normal(0, 1, size=64),
            "z": rng.normal(0, 1, size=64),
        }
    )
    # And then disappears again.
    yield pd.DataFrame({"x": rng.normal(0, 1, size=64), "y": rng.normal(0, 1, size=64)})


batch_idx = 0
async for result in monitor.monitor(schema_change_stream()):
    print(f"batch {batch_idx}: scored columns = {sorted(result.keys())}")
    batch_idx += 1

## What's next

- For real broker integrations, see `KafkaStreamMonitor` and `RabbitMQStreamMonitor` —
  same surface, broker-decoded `pd.read_json` per message.
- For multiple-detector consensus per batch, route the detector through `EnsembleDriftDetector`
  (see `04_ensemble_and_multiple_testing.ipynb`).